# LeetCode 3950 — Exactly One Consecutive Set Bits Pair bằng Python

Link đề: https://leetcode.com/problems/exactly-one-consecutive-set-bits-pair/ · Độ khó: **Easy** · Chủ đề: Bit Manipulation, Math

Skeleton LeetCode:

```python
class Solution:
    def consecutiveSetBits(self, n: int) -> bool:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **cặp bit 1 đứng cạnh nhau** (hai vị trí liền kề cùng bằng `1`) trong biểu diễn nhị phân của `n`; trả `True` nếu có **đúng một** cặp.
2. Biết các toán tử bit: `n & 1`, `n >> 1`, `n & (n >> 1)`, `bin(n)`, `bit_count` / `bin(x).count('1')`.
3. Nhận ra **bước đột phá**: các cặp liền kề tương ứng với các bit `1` của `n & (n >> 1)`; chỉ cần kiểm tra giá trị đó có **đúng một bit 1** (lũy thừa của 2).
4. Viết 4 cách (quét chuỗi nhị phân, quét bit, `popcount`, mẹo `x & (x - 1)`) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: dùng `str.count('11')` (không chồng lấn), kiểm tra `'11' in bin(n)` (có ít nhất một cặp), quên `n = 0`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Hãy trả về `True` nếu biểu diễn nhị phân của `n` có **đúng một cặp bit 1 liền kề**, ngược lại `False`. "Cặp liền kề" gồm hai vị trí bit cạnh nhau cùng bằng `1`; các cặp được tính **chồng lấn** (ví dụ `111` có hai cặp: bit 0-1 và bit 1-2).

| Ví dụ | `n` | nhị phân | Output | Giải thích |
|---|---|---|---|---|
| 1 | `6` | `110` | `True` | Chỉ có một cặp `11` |
| 2 | `5` | `101` | `False` | Không có hai bit `1` liền kề |
| (tự thêm) | `7` | `111` | `False` | Có **hai** cặp (chồng lấn) |
| (tự thêm) | `15` | `1111` | `False` | Có **ba** cặp |

Constraints: `0 <= n <= 10^5`.

In [1]:
for n in (6, 5, 7, 15, 12, 0):
    b = bin(n)[2:]
    pairs = sum(1 for i in range(len(b) - 1) if b[i] == b[i + 1] == "1")
    print(f"n={n:>3}  nhị phân={b:>5}  số cặp liền kề={pairs}  -> {pairs == 1}")

n=  6  nhị phân=  110  số cặp liền kề=1  -> True
n=  5  nhị phân=  101  số cặp liền kề=0  -> False
n=  7  nhị phân=  111  số cặp liền kề=2  -> False
n= 15  nhị phân= 1111  số cặp liền kề=3  -> False
n= 12  nhị phân= 1100  số cặp liền kề=1  -> True
n=  0  nhị phân=    0  số cặp liền kề=0  -> False


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân và `bin`

`bin(6)` cho chuỗi `'0b110'`; bỏ tiền tố `'0b'` bằng `bin(6)[2:]` hoặc `format(6, 'b')`. Bit thấp nhất (phải nhất) là bit 0.

In [2]:
print(bin(6), bin(6)[2:], format(6, "b"), bin(0))

0b110 110 110 0b0


### 3.2. Dịch bit `>>` và `<<`

`n >> 1` dịch sang phải 1 bit (bỏ bit thấp nhất, chia nguyên cho 2). `n << 1` dịch trái (nhân 2). Dịch sang phải làm **bit ở vị trí `i + 1` rơi xuống vị trí `i`**: nhờ vậy `n` và `n >> 1` đặt **hai bit liền kề thẳng hàng**.

In [3]:
n = 6                                        # 110
print(bin(n), bin(n >> 1), bin(n << 1))

0b110 0b11 0b1100


### 3.3. Toán tử AND `&`

`a & b` chỉ giữ bit `1` ở những vị trí **cả hai** đều là `1`. `n & 1` lấy bit thấp nhất. `n & (n >> 1)` có bit `1` ở vị trí `i` đúng khi bit `i` và bit `i + 1` của `n` đều là `1`.

In [4]:
for n in (6, 7, 5):
    m = n & (n >> 1)
    print(f"n={n} ({n:03b}), n>>1 ({n >> 1:03b}), n&(n>>1) = {m:03b} = {m}")

n=6 (110), n>>1 (011), n&(n>>1) = 010 = 2
n=7 (111), n>>1 (011), n&(n>>1) = 011 = 3
n=5 (101), n>>1 (010), n&(n>>1) = 000 = 0


### 3.4. Đếm bit 1 (popcount)

`bin(x).count('1')` đếm số bit `1`; từ Python 3.10 có `x.bit_count()`. Notebook dùng `bin(x).count('1')` để chạy được trên mọi phiên bản Python 3.

In [5]:
print(bin(7).count("1"), bin(8).count("1"), bin(0).count("1"))

3 1 0


### 3.5. Mẹo `x & (x - 1)`: xóa bit 1 thấp nhất

`x & (x - 1)` xóa bit `1` thấp nhất của `x`. Hệ quả: `x > 0` có **đúng một bit 1** (là lũy thừa của 2) khi và chỉ khi `x & (x - 1) == 0`.

In [6]:
for x in (1, 2, 3, 4, 6, 8, 12):
    print(f"x={x:>2} ({x:04b})  x&(x-1)={x & (x - 1):>2}  lũy thừa 2? {x > 0 and x & (x - 1) == 0}")

x= 1 (0001)  x&(x-1)= 0  lũy thừa 2? True
x= 2 (0010)  x&(x-1)= 0  lũy thừa 2? True
x= 3 (0011)  x&(x-1)= 2  lũy thừa 2? False
x= 4 (0100)  x&(x-1)= 0  lũy thừa 2? True
x= 6 (0110)  x&(x-1)= 4  lũy thừa 2? False
x= 8 (1000)  x&(x-1)= 0  lũy thừa 2? True
x=12 (1100)  x&(x-1)= 8  lũy thừa 2? False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách nghĩ trực tiếp nhất: viết `n` ra nhị phân, rồi **đi qua từng cặp vị trí cạnh nhau** và đếm các cặp `11`.

Tính tay `n = 12 = 1100`: các cặp liền kề là `(1,1)`, `(1,0)`, `(0,0)` -> chỉ một cặp `11` -> `True`.
Tính tay `n = 13 = 1101`: cặp `(1,1)`, `(1,0)`, `(0,1)` -> một cặp `11` -> `True`.
Tính tay `n = 14 = 1110`: cặp `(1,1)`, `(1,1)`, `(1,0)` -> **hai** cặp -> `False`.

| `n` | nhị phân | các cặp `11` | kết quả |
|---|---|---|---|
| 12 | `1100` | 1 | `True` |
| 13 | `1101` | 1 | `True` |
| 14 | `1110` | 2 | `False` |
| 10 | `1010` | 0 | `False` |

In [7]:
def pairs_of(n):
    b = format(n, "b")
    return sum(1 for i in range(len(b) - 1) if b[i] == "1" and b[i + 1] == "1")

for n in (12, 13, 14, 10):
    print(n, format(n, "b"), pairs_of(n), pairs_of(n) == 1)

12 1100 1 True
13 1101 1 True
14 1110 2 False
10 1010 0 False


## 5. Cách 1 — Brute Force / baseline: quét chuỗi nhị phân

1. `b = bin(n)[2:]` (với `n = 0` ta có `'0'`, không có cặp nào).
2. Với mỗi `i` từ `0` đến `len(b) - 2`, nếu `b[i] == b[i+1] == '1'` thì tăng đếm.
3. Trả `đếm == 1`.

Đúng và dễ hiểu; "nút thắt" là phải **dựng chuỗi** và duyệt từng cặp. Có thể làm gọn hơn nếu biết cách để máy tự đếm cặp bằng phép AND.

In [8]:
class SolutionString:
    def consecutiveSetBits(self, n):
        b = bin(n)[2:]
        pairs = 0
        for i in range(len(b) - 1):
            if b[i] == "1" and b[i + 1] == "1":
                pairs += 1
        return pairs == 1

print(SolutionString().consecutiveSetBits(6), SolutionString().consecutiveSetBits(5), SolutionString().consecutiveSetBits(7))

True False False


**Độ phức tạp:** Time `O(log n)` (số bit), Space `O(log n)` cho chuỗi.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đặt `m = n & (n >> 1)`. Bit `i` của `m` bằng `1` **khi và chỉ khi** bit `i` và bit `i + 1` của `n` đều là `1`, tức vị trí `i` là **đầu dưới của một cặp liền kề**. Vì vậy

`số cặp liền kề của n  =  số bit 1 của m`.

Điều kiện "đúng một cặp" trở thành **"`m` có đúng một bit 1"**, tức `m` là lũy thừa của 2: `m > 0 and m & (m - 1) == 0`.

**Quy tắc "nếu... thì...":** nếu `m == 0` thì không có cặp nào -> `False`; nếu `m` có từ hai bit `1` trở lên thì có `>= 2` cặp -> `False`; chỉ khi `m` là lũy thừa của 2 thì `True`.

**Invariant:** các chỉ số bit mà `m` có giá trị `1` chính xác là danh sách "chỉ số thấp của mọi cặp `11`"; không bị chồng lấn hay bỏ sót, vì mỗi cặp `(i, i+1)` được nhận diện bằng đúng một vị trí `i`.

## 7. Các cách tiếp theo

### Cách 2 — Quét từng bit bằng `n & 1` và `n >>= 1`

Không dùng chuỗi: giữ `prev` (bit trước), mỗi bước đọc bit hiện tại `cur`; nếu `prev == cur == 1` thì đếm một cặp.

In [9]:
class SolutionScan:
    def consecutiveSetBits(self, n):
        pairs = 0
        prev = 0
        while n:
            cur = n & 1
            if prev and cur:
                pairs += 1
            prev = cur
            n >>= 1
        return pairs == 1

print(SolutionScan().consecutiveSetBits(6), SolutionScan().consecutiveSetBits(5), SolutionScan().consecutiveSetBits(7))

True False False


**Độ phức tạp:** Time `O(log n)`, Space `O(1)`.

### Cách 3 — `popcount(n & (n >> 1)) == 1`

Một dòng, đúng theo bước đột phá: đếm số bit `1` của `n & (n >> 1)`.

In [10]:
class SolutionPopcount:
    def consecutiveSetBits(self, n):
        return bin(n & (n >> 1)).count("1") == 1

print(SolutionPopcount().consecutiveSetBits(6), SolutionPopcount().consecutiveSetBits(5), SolutionPopcount().consecutiveSetBits(7))

True False False


**Độ phức tạp:** Time `O(log n)` (đếm bit), Space `O(log n)` cho chuỗi trung gian của `bin`.

### Cách 4 — Mẹo `m & (m - 1) == 0` (không đếm, không chuỗi)

`m = n & (n >> 1)`; trả `m != 0 and m & (m - 1) == 0`. Chỉ vài phép toán bit, `O(1)` thực tế (số nguyên 17 bit).

In [11]:
class SolutionBitTrick:
    def consecutiveSetBits(self, n):
        m = n & (n >> 1)
        return m != 0 and (m & (m - 1)) == 0

print(SolutionBitTrick().consecutiveSetBits(6), SolutionBitTrick().consecutiveSetBits(5), SolutionBitTrick().consecutiveSetBits(7))

True False False


**Độ phức tạp:** Time `O(1)` (một số hạng phép toán bit trên số nguyên nhỏ), Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 trên `n = 14 = 1110` (hai cặp, kết quả `False`):

| bước | `n` (nhị phân) | `cur` | `prev` | cặp `11`? | `pairs` |
|---|---|---|---|---|---|
| 1 | `1110` | 0 | 0 | không | 0 |
| 2 | `111` | 1 | 0 | không | 0 |
| 3 | `11` | 1 | 1 | **có** | 1 |
| 4 | `1` | 1 | 1 | **có** | 2 |

Với `n = 6 = 110`: bit `0, 1, 1` -> chỉ bước cuối có `prev = cur = 1` -> `pairs = 1` -> `True`.

In [12]:
def trace_scan(n):
    print(f"n={n} ({n:b})")
    pairs, prev, step = 0, 0, 0
    while n:
        step += 1
        cur = n & 1
        hit = bool(prev and cur)
        if hit:
            pairs += 1
        print(f"  bước {step}: n={n:b}, cur={cur}, prev={prev}, cặp 11? {hit}, pairs={pairs}")
        prev = cur
        n >>= 1
    print("  ->", pairs == 1)
    return pairs == 1

assert trace_scan(14) is False
assert trace_scan(6) is True

n=14 (1110)
  bước 1: n=1110, cur=0, prev=0, cặp 11? False, pairs=0
  bước 2: n=111, cur=1, prev=0, cặp 11? False, pairs=0
  bước 3: n=11, cur=1, prev=1, cặp 11? True, pairs=1
  bước 4: n=1, cur=1, prev=1, cặp 11? True, pairs=2
  -> False
n=6 (110)
  bước 1: n=110, cur=0, prev=0, cặp 11? False, pairs=0
  bước 2: n=11, cur=1, prev=0, cặp 11? False, pairs=0
  bước 3: n=1, cur=1, prev=1, cặp 11? True, pairs=1
  -> True


In [13]:
def trace_and(n):
    m = n & (n >> 1)
    print(f"n   = {n:08b}\nn>>1= {n >> 1:08b}\nm   = {m:08b}  (các vị trí bit i mà bit i, i+1 của n đều là 1)")
    ok = m != 0 and (m & (m - 1)) == 0
    print("  m & (m-1) =", m & (m - 1) if m else "n/a", "->", ok)
    return ok

assert trace_and(6) is True and trace_and(14) is False and trace_and(5) is False

n   = 00000110
n>>1= 00000011
m   = 00000010  (các vị trí bit i mà bit i, i+1 của n đều là 1)
  m & (m-1) = 0 -> True
n   = 00001110
n>>1= 00000111
m   = 00000110  (các vị trí bit i mà bit i, i+1 của n đều là 1)
  m & (m-1) = 4 -> False
n   = 00000101
n>>1= 00000010
m   = 00000000  (các vị trí bit i mà bit i, i+1 của n đều là 1)
  m & (m-1) = n/a -> False


## 9. Code nộp LeetCode

Dùng Cách 4 (mẹo bit, ngắn và không dùng chuỗi).

In [14]:
from typing import List, Optional


class Solution:
    def consecutiveSetBits(self, n: int) -> bool:
        m = n & (n >> 1)
        return m != 0 and (m & (m - 1)) == 0


print(Solution().consecutiveSetBits(6), Solution().consecutiveSetBits(5), Solution().consecutiveSetBits(7))

True False False


## 10. Vì sao thuật toán đúng

1. **Ánh xạ cặp -> bit.** Gọi `b_i` là bit thứ `i` của `n` (`b_i = 0` khi vượt độ dài). Bit thứ `i` của `n >> 1` là `b_{i+1}`. Nên bit thứ `i` của `m = n & (n >> 1)` là `b_i AND b_{i+1}`.
2. **Một-một.** Cặp liền kề `(i, i+1)` có cả hai bit bằng `1` khi và chỉ khi bit `i` của `m` bằng `1`. Hai cặp khác nhau có chỉ số `i` khác nhau, nên mỗi cặp tương ứng đúng **một** bit `1` của `m`.
3. **Đếm.** Vậy số cặp liền kề = `popcount(m)`.
4. **Điều kiện đúng một cặp** <=> `popcount(m) == 1` <=> `m` là lũy thừa của 2 <=> `m > 0` và `m & (m - 1) == 0` (vì `m & (m - 1)` xóa bit `1` thấp nhất, kết quả `0` chỉ khi không còn bit `1` nào khác).
5. **Trường hợp `n = 0`:** `m = 0`, không có cặp -> `False`, khớp với phép kiểm tra `m != 0`.

Chuỗi (Cách 1), quét bit (Cách 2), `popcount` (Cách 3) đều đếm cùng đại lượng theo cách khác nhau.

## 11. Phân tích Time / Space Complexity

Gọi `B = số bit của n` (`B <= 17` vì `n <= 10^5 < 2^17`).

- Cách 1 (chuỗi): Time `O(B)`, Space `O(B)`.
- Cách 2 (quét bit): Time `O(B)`, Space `O(1)`.
- Cách 3 (`popcount`): Time `O(B)`, Space `O(B)` (chuỗi của `bin`).
- Cách 4 (nộp): Time `O(1)` phép bit (số nguyên máy), Space `O(1)`.

Trong ngôn ngữ có số nguyên cố định 32 bit, Cách 4 chỉ vài lệnh CPU. Với Python, mỗi phép bit trên số nhỏ cũng là hằng số.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionString(), SolutionScan(), SolutionPopcount(), SolutionBitTrick(), Solution()]

fixed = [
    (6, True), (5, False), (0, False), (1, False), (2, False), (3, True), (7, False),
    (12, True), (13, True), (14, False), (15, False), (24, True), (48, True), (96, True),
    (100000, False), (99999, False), (98304, True), (65536, False), (65535, False),
]
for n, expected in fixed:
    for sol in solutions:
        assert sol.consecutiveSetBits(n) == expected, (type(sol).__name__, n)
print("test cố định OK:", len(fixed), "ca")

# vét cạn toàn bộ miền của đề: 0..10^5
for n in range(0, 10 ** 5 + 1):
    expected = pairs_of(n) == 1
    assert Solution().consecutiveSetBits(n) == expected, n
print("vét cạn 0..10^5 OK")

random.seed(3950)
for _ in range(500):
    n = random.randint(0, 10 ** 5)
    outs = {sol.consecutiveSetBits(n) for sol in solutions}
    assert len(outs) == 1, n
print("test ngẫu nhiên OK")

test cố định OK: 19 ca
vét cạn 0..10^5 OK
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) `n` và `n * 2` có cùng số cặp (dịch trái không đổi khoảng cách bit) nên cùng đáp án. (2) Số cặp của `n` (`popcount(n & (n >> 1))`) khớp với đếm trên chuỗi. (3) Số dạng `2^k * 3` (tức `11` rồi `k` bit `0`) luôn `True`.

In [16]:
random.seed(39500)
for _ in range(500):
    n = random.randint(0, 5 * 10 ** 4)
    assert Solution().consecutiveSetBits(n) == Solution().consecutiveSetBits(n * 2)
    assert bin(n & (n >> 1)).count("1") == pairs_of(n)
for k in range(0, 15):
    assert Solution().consecutiveSetBits(3 << k) is True
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `str.count("11")`

`"111".count("11")` chỉ cho `1` vì `count` tìm **không chồng lấn**. Nhưng `7 = 111` có **hai** cặp chồng lấn, nên đáp án đúng là `False`.

In [17]:
def wrong_count_11(n):
    return bin(n)[2:].count("11") == 1                       # SAI: count không chồng lấn

print(wrong_count_11(7), "!=", Solution().consecutiveSetBits(7))
assert wrong_count_11(7) is True and Solution().consecutiveSetBits(7) is False

True != False


### Lỗi 2: kiểm tra `"11" in bin(n)` (có ít nhất một cặp)

Đề yêu cầu **đúng một** cặp. `15 = 1111` chứa `11` nhưng có tới ba cặp.

In [18]:
def wrong_at_least_one(n):
    return "11" in bin(n)                                    # SAI: "ít nhất một", không phải "đúng một"

print(wrong_at_least_one(15), "!=", Solution().consecutiveSetBits(15))
assert wrong_at_least_one(15) is True and Solution().consecutiveSetBits(15) is False

True != False


### Lỗi 3: kiểm tra `n & (n >> 1) == 1`

Chỉ đúng khi cặp nằm ở bit thấp nhất. `12 = 1100` có cặp ở vị trí 2: `m = 4`, không bằng `1`, nhưng đáp án là `True`.

In [19]:
def wrong_eq_one(n):
    return (n & (n >> 1)) == 1                               # SAI: m có thể là 2, 4, 8,...

print(wrong_eq_one(12), "!=", Solution().consecutiveSetBits(12))
assert wrong_eq_one(12) is False and Solution().consecutiveSetBits(12) is True
assert wrong_eq_one(3) is True

False != True


### Lỗi 4: quên `m != 0` (với `n = 0` hoặc `n` không có cặp nào)

`m & (m - 1) == 0` cũng đúng khi `m = 0` (`0 & -1 = 0`). Thiếu `m != 0` sẽ trả `True` nhầm cho `n = 5`.

In [20]:
def wrong_forgot_nonzero(n):
    m = n & (n >> 1)
    return (m & (m - 1)) == 0                                # SAI: m = 0 cũng thỏa

print(wrong_forgot_nonzero(5), "!=", Solution().consecutiveSetBits(5))
assert wrong_forgot_nonzero(5) is True and wrong_forgot_nonzero(0) is True
assert Solution().consecutiveSetBits(5) is False and Solution().consecutiveSetBits(0) is False

True != False


### Lỗi 5: nhầm `>>` với `<<` hoặc nhầm độ ưu tiên

`n & n >> 1` được Python hiểu là `n & (n >> 1)` (đúng vì `>>` ưu tiên hơn `&`) nhưng dễ gây hiểu lầm; luôn đặt ngoặc. Dùng `<<` thay `>>` cho kết quả có ý nghĩa tương tự (`n & (n << 1)`) nhưng bit chỉ số `i + 1`; vẫn cùng số bit 1 nên **không sai** về số lượng. Lỗi thật là so sánh `n & (n >> 1) == 0`, chỉ kiểm tra "không có cặp nào".

In [21]:
def wrong_zero_pairs(n):
    return (n & (n >> 1)) == 0                               # SAI: đây là "không có cặp nào"

print(wrong_zero_pairs(6), "!=", Solution().consecutiveSetBits(6))
assert wrong_zero_pairs(6) is False and Solution().consecutiveSetBits(6) is True
assert bin(6 & (6 << 1)).count("1") == bin(6 & (6 >> 1)).count("1")    # << cũng đếm đúng số cặp

False != True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai bit liền kề đều là 1" -> `n & (n >> 1)`. Mẫu này cũng dùng để kiểm tra **không có hai bit 1 liền kề** (`n & (n >> 1) == 0`, bài 693) và để đếm cặp.
- "Đúng một bit 1" -> lũy thừa của 2: `x > 0 and x & (x - 1) == 0`.
- Nhớ `>>` ưu tiên hơn `&`, nhưng **luôn đặt ngoặc** cho dễ đọc.
- Với ràng buộc nhỏ (`n <= 10^5`) có thể **vét cạn** toàn miền để kiểm chứng (như bài test ở trên): thủ thuật tự kiểm tra rất mạnh.
- Cẩn thận hiểu đề: cặp là **chồng lấn** hay không? Tự thêm ví dụ `7 = 111` để tránh hiểu lầm.
- `n = 0` luôn cần được thử riêng.

## 15. Biến thể của bài

1. **Binary Number with Alternating Bits** (693): kiểm tra mọi hai bit liền kề đều khác nhau.
2. **Number of 1 Bits** (191), **Counting Bits** (338): popcount.
3. **Power of Two** (231): `n & (n - 1) == 0`.
4. **Binary Gap** (868): khoảng cách lớn nhất giữa hai bit 1.
5. Biến thể: yêu cầu **đúng `k` cặp** liền kề: so `popcount(n & (n >> 1)) == k`.
6. Biến thể: đếm các số trong `[0, N]` có đúng một cặp liền kề (vét cạn cho `N` nhỏ, DP chữ số cho `N` lớn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét chuỗi | đếm cặp `11` trong `bin(n)` | `O(B)` | `O(B)` | Dễ hiểu, dễ gỡ lỗi |
| 2. Quét bit | `prev`, `cur` với `& 1`, `>>= 1` | `O(B)` | `O(1)` | Không dùng chuỗi |
| 3. `popcount` | `bin(n & (n >> 1)).count('1') == 1` | `O(B)` | `O(B)` | Một dòng, rõ ý nghĩa |
| 4. Mẹo bit | `m != 0 and m & (m - 1) == 0` | `O(1)` | `O(1)` | **Nộp bài** |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().consecutiveSetBits(11)` và `Solution().consecutiveSetBits(28)` ra gì? (viết nhị phân trước).

**Bài 2 (tính tay):** với `n = 100000`, viết nhị phân, tìm `n & (n >> 1)`, đếm số cặp.

**Bài 3 (điền TODO):** hoàn thành `count_adjacent_pairs(n)` trả về **số cặp** bit 1 liền kề (không chỉ True/False).

**Bài 4 (tự viết lại):** viết `has_no_adjacent_ones(n)` (không có cặp nào, bài 693 ngược lại) bằng đúng một biểu thức bit.

**Bài 5 (mini challenge):** đếm có bao nhiêu `n` trong `[0, 100]` có đúng một cặp liền kề, rồi in vài số nhỏ nhất.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().consecutiveSetBits(11), Solution().consecutiveSetBits(28))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_adjacent_pairs_todo(n):
    # TODO: số bit 1 của n & (n >> 1)
    return 0

True False


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `11 = 1011`, `28 = 11100`. Đếm các cặp `11`. Bài 2: viết nhị phân của `100000`.

**Mức 2 (vừa):** Bài 3: `bin(n & (n >> 1)).count('1')`. Bài 4: `n & (n >> 1) == 0`. Bài 5: vòng `for n in range(101)` rồi lọc.

**Mức 3 (gần đáp án):** Bài 1: `11` có một cặp (`10 11`) -> `True`; `28 = 11100` có hai cặp -> `False`. Bài 2: `100000 = 11000011010100000` có một cặp ở đầu và một ở giữa -> `False`. Bài 3: `15 -> 3`. Bài 5: bắt đầu `3, 6, 11, 12, 13, ...`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().consecutiveSetBits(11) is True and Solution().consecutiveSetBits(28) is False
assert format(100000, "b") == "11000011010100000"
assert bin(100000 & (100000 >> 1)).count("1") == 2 and Solution().consecutiveSetBits(100000) is False

# Bài 3
def count_adjacent_pairs(n):
    return bin(n & (n >> 1)).count("1")

assert count_adjacent_pairs(15) == 3 and count_adjacent_pairs(7) == 2 and count_adjacent_pairs(5) == 0

# Bài 4
def has_no_adjacent_ones(n):
    return (n & (n >> 1)) == 0

assert has_no_adjacent_ones(5) is True and has_no_adjacent_ones(6) is False and has_no_adjacent_ones(0) is True

# Bài 5
good = [n for n in range(101) if Solution().consecutiveSetBits(n)]
assert good[:5] == [3, 6, 11, 12, 13] and len(good) == sum(1 for n in range(101) if count_adjacent_pairs(n) == 1)
print(len(good), good[:10])
print("đáp án bài tập OK")

34 [3, 6, 11, 12, 13, 19, 22, 24, 25, 26]
đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def consecutiveSetBits(self, n: int) -> bool:
        m = n & (n >> 1)                      # bit i của m = 1  <=>  bit i và i+1 của n đều là 1
        return m != 0 and (m & (m - 1)) == 0  # m có đúng một bit 1
```

Khung chung: "đếm cặp bit liền kề" -> `popcount(n & (n >> 1))`; "đúng một bit 1" -> `m & (m - 1) == 0`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Đã quyết định cặp liền kề tính **chồng lấn** (`111` có 2 cặp) và thử `n = 7`, `n = 0`.
- [ ] Có kiểm tra `m != 0`; dùng ngoặc cho `n & (n >> 1)`.